# MCTS v4 — Stockfish Teacher

ResNet(768, 5 blocks, 128 filters) trained via MCTS + Stockfish at fixed ELO.

Only engine half-moves enter the replay buffer (MCTS visit counts → policy target; Stockfish half-moves excluded).

In [ ]:
# Imports
import subprocess, sys, asyncio, glob
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'chess', 'torch'], check=False)

import os, sqlite3, time, random
from collections import deque
from datetime import datetime

import chess
import chess.engine
import chess.pgn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import MultiStepLR

import engine_nn_v2 as eng

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

PLANE_NAMES = eng.PLANE_NAMES
print(f'Plane names ({len(PLANE_NAMES)}): {PLANE_NAMES}')

In [ ]:
# Configuration
RUN_ID             = 34
N_GAMES            = 800
N_SIMULATIONS      = 200          # MCTS simulations per engine move
SF_ELO             = 1320         # Stockfish ELO (minimum 1320 for Stockfish 19)
REPLAY_BUFFER_SIZE = 10_000
BATCH_SIZE         = 64
LR                 = 1e-3
WEIGHT_DECAY       = 1e-4
LR_MILESTONES      = [200, 500]   # absolute game counts where LR drops 10x
LR_GAMMA           = 0.1
TEMPERATURE        = 1.0          # MCTS visit-count temperature during training
CHECKPOINT_EVERY   = 10
VALUE_LOSS_WEIGHT  = 2.0         # weight on value loss relative to policy loss
RESIGN_THRESHOLD   = 0.9
RESIGN_CONSECUTIVE = 4
RESUME             = True

DB_PATH    = '../../data/chess_learning.db'
MODEL_DIR  = '../../models'
MODEL_PATH = f'{MODEL_DIR}/mcts_sf_v4_run_{RUN_ID}.pt'

In [ ]:
# Database setup
con = sqlite3.connect(DB_PATH)
cur = con.cursor()

try:
    cur.execute('ALTER TABLE game_records ADD COLUMN policy_loss REAL')
    con.commit()
    print('Added policy_loss column')
except Exception:
    pass

NOTES          = (f'mcts_sf_v4_ResNet_768bitboard_5blocks_128filters '
                  f'n_simulations={N_SIMULATIONS} sf_elo={SF_ELO} '
                  f'batch={BATCH_SIZE} replay={REPLAY_BUFFER_SIZE}')
ENGINE_VERSION = 'engine_nn_v2'

existing = cur.execute('SELECT run_id FROM runs WHERE run_id = ?', (RUN_ID,)).fetchone()
if existing and not RESUME:
    raise RuntimeError(f'run_id={RUN_ID} already exists. Set RESUME=True or choose a new RUN_ID.')
if not existing:
    cur.execute(
        'INSERT INTO runs (run_id, started_at, depth, learning_rate, move_limit, '
        'notes, last_updated, engine_version) VALUES (?, ?, ?, ?, ?, ?, ?, ?)',
        (RUN_ID, datetime.now().isoformat(), 0, LR, None,
         NOTES, datetime.now().isoformat(), ENGINE_VERSION)
    )
    con.commit()
    print(f'Created run_id={RUN_ID}')
else:
    print(f'Resuming run_id={RUN_ID}')

row        = cur.execute('SELECT MAX(game_number) FROM games WHERE run_id = ?', (RUN_ID,)).fetchone()[0]
start_game = (row or 0) + 1
print(f'Starting at game_number={start_game}')

In [ ]:
# Network, optimizer, replay buffer
os.makedirs(MODEL_DIR, exist_ok=True)

if RESUME and os.path.exists(MODEL_PATH):
    net = eng.load_model(MODEL_PATH, net_class=eng.ResNet,
                         input_dim=768, n_filters=128, n_blocks=5)
    print(f'Loaded model from {MODEL_PATH}')
else:
    net = eng.ResNet(input_dim=768, n_filters=128, n_blocks=5)
    n_params = sum(p.numel() for p in net.parameters())
    print(f'Initialized fresh ResNet(768, 5 blocks, 128 filters) -- {n_params:,} parameters')

optimizer = optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

session_milestones = [max(1, m - start_game + 1) for m in LR_MILESTONES if m >= start_game]
scheduler = MultiStepLR(optimizer, milestones=session_milestones, gamma=LR_GAMMA)
print(f'LR milestones (session-relative): {session_milestones}')

# Replay buffer: engine positions only
# Each example: (board_tensor, value_target, legal_moves, policy_target)
replay_buffer = deque(maxlen=REPLAY_BUFFER_SIZE)

In [ ]:
# Stockfish setup
def find_stockfish():
    local_app = os.environ.get('LOCALAPPDATA', '')
    pattern   = os.path.join(
        local_app, 'Microsoft', 'WinGet', 'Packages', 'Stockfish*', '**', 'stockfish*.exe'
    )
    hits = glob.glob(pattern, recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError('Stockfish not found. Install with: winget install Stockfish.Stockfish')

SF_PATH = find_stockfish()
print(f'Stockfish found: {SF_PATH}')

In [ ]:
# Game vs Stockfish: engine uses MCTS, SF plays at fixed ELO
def play_game_vs_sf(net, sf, engine_color,
                    n_simulations=N_SIMULATIONS, temperature=TEMPERATURE,
                    resign_threshold=RESIGN_THRESHOLD, resign_consecutive=RESIGN_CONSECUTIVE):
    '''
    Play one game: engine (ResNet + MCTS) vs Stockfish.
    Only engine half-moves are collected as training examples.
    Returns: examples, board, n_halfmoves, terminated_by, resigned_turn
    '''
    board           = chess.Board()
    positions       = []   # engine turns: (tensor, turn, legal_moves, policy_target)
    n_halfmoves     = 0
    consecutive_bad = 0
    terminated_by   = 'natural'
    resigned_turn   = None

    net.eval()
    while not board.is_game_over(claim_draw=True):
        if board.turn == engine_color:
            tensor = eng.board_to_tensor(board)

            # Resignation check (ResNet returns policy, value)
            with torch.no_grad():
                _, val_out = net(tensor.unsqueeze(0))
                lv = val_out.squeeze().item()
            if lv < -resign_threshold:
                consecutive_bad += 1
                if consecutive_bad >= resign_consecutive:
                    terminated_by = 'resignation'
                    resigned_turn = board.turn
                    break
            else:
                consecutive_bad = 0

            legal_moves = list(board.legal_moves)
            move, visit_counts = eng.mcts_search(
                board, net,
                n_simulations=n_simulations,
                temperature=temperature,
            )
            total         = sum(visit_counts.values()) or 1
            policy_target = torch.tensor(
                [visit_counts.get(m, 0) / total for m in legal_moves],
                dtype=torch.float32
            )
            positions.append((tensor, board.turn, legal_moves, policy_target))

        else:
            result = sf.play(board, chess.engine.Limit(time=0.5), ponder=False)
            if result.move is None:
                break
            move = result.move
            consecutive_bad = 0

        board.push(move)
        n_halfmoves += 1

    if terminated_by == 'resignation':
        white_val = -1.0 if resigned_turn == chess.WHITE else 1.0
    else:
        result    = board.result(claim_draw=True)
        white_val = {'1-0': 1.0, '0-1': -1.0}.get(result, 0.0)

    examples = [
        (tensor,
         white_val if turn == chess.WHITE else -white_val,
         legal_moves,
         policy_target)
        for tensor, turn, legal_moves, policy_target in positions
    ]
    return examples, board, n_halfmoves, terminated_by, resigned_turn

In [ ]:
# Dual-loss training step (value loss upweighted by VALUE_LOSS_WEIGHT)
def train_step(net, optimizer, batch):
    net.train()
    optimizer.zero_grad()

    total_value_loss  = torch.tensor(0.0)
    total_policy_loss = torch.tensor(0.0)

    for board_tensor, value_target, legal_moves, policy_target in batch:
        policy_probs, value = net.forward(board_tensor, legal_moves)

        v_loss = F.mse_loss(value, torch.tensor(value_target, dtype=torch.float32))
        p_loss = -(policy_target * torch.log(policy_probs.clamp(min=1e-8))).sum()

        total_value_loss  = total_value_loss  + v_loss
        total_policy_loss = total_policy_loss + p_loss

    n    = len(batch)
    loss = (VALUE_LOSS_WEIGHT * total_value_loss + total_policy_loss) / n
    loss.backward()
    nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
    optimizer.step()

    return loss.item(), (total_value_loss / n).item(), (total_policy_loss / n).item()


def eval_policy_loss(net, examples):
    if not examples:
        return 0.0
    net.eval()
    total = 0.0
    with torch.no_grad():
        for board_tensor, _, legal_moves, policy_target in examples:
            policy_probs, _ = net.forward(board_tensor, legal_moves)
            total += -(policy_target * torch.log(policy_probs.clamp(min=1e-8))).sum().item()
    return total / len(examples)

In [ ]:
# Logging helpers
def board_to_pgn_str(board, game_number, engine_color):
    game = chess.pgn.Game.from_board(board)
    game.headers['Event'] = 'MCTS v4 SF Teacher'
    game.headers['Round'] = str(game_number)
    game.headers['White'] = 'Engine' if engine_color == chess.WHITE else 'Stockfish'
    game.headers['Black'] = 'Stockfish' if engine_color == chess.WHITE else 'Engine'
    return str(game)


def log_game(game_number, board, n_halfmoves, duration_s, policy_loss,
             engine_color, terminated_by='natural', resigned_turn=None):
    if terminated_by == 'resignation':
        outcome_str = '0-1' if resigned_turn == chess.WHITE else '1-0'
    else:
        result      = board.result(claim_draw=True)
        outcome_str = {'1-0': '1-0', '0-1': '0-1'}.get(result, '1/2-1/2')
    # Store SF color in human_color field (mirrors human-play convention)
    sf_color_str = 'White' if engine_color == chess.BLACK else 'Black'
    cur.execute(
        'INSERT INTO games (run_id, game_number, outcome, n_halfmoves, duration_s, '
        'terminated_by, human_color) VALUES (?, ?, ?, ?, ?, ?, ?)',
        (RUN_ID, game_number, outcome_str, n_halfmoves,
         round(duration_s, 2), terminated_by, sf_color_str)
    )
    cur.execute(
        'INSERT INTO game_records (run_id, game_number, pgn, policy_loss) VALUES (?, ?, ?, ?)',
        (RUN_ID, game_number,
         board_to_pgn_str(board, game_number, engine_color), round(policy_loss, 6))
    )
    con.commit()
    return outcome_str


def log_weights(game_number, prev_weights):
    attr        = eng.attribution_weights(chess.Board(), net)
    scalar_attr = {k: v for k, v in attr.items() if isinstance(v, float)}
    for name, val in scalar_attr.items():
        cur.execute(
            'INSERT INTO weights (run_id, game_number, feature_name, weight_value) '
            'VALUES (?,?,?,?)',
            (RUN_ID, game_number, name, round(float(val), 6))
        )
        if prev_weights and name in prev_weights:
            cur.execute(
                'INSERT INTO weight_deltas (run_id, game_number, feature_name, delta) '
                'VALUES (?,?,?,?)',
                (RUN_ID, game_number, name, round(float(val) - prev_weights[name], 6))
            )
    con.commit()
    return scalar_attr


def log_feature_stats(game_number, outcome_str, examples):
    if not examples:
        return
    stacked     = torch.stack([x[0] for x in examples]).view(-1, 12, 64)
    plane_means = stacked.mean(dim=0).sum(dim=1).tolist()
    cur.executemany(
        'INSERT INTO feature_stats (run_id, game_number, outcome, feature_name, mean_value) '
        'VALUES (?,?,?,?,?)',
        [(RUN_ID, game_number, outcome_str, name, round(plane_means[i], 6))
         for i, name in enumerate(PLANE_NAMES)]
    )
    con.commit()

In [ ]:
# Training loop
sf = chess.engine.SimpleEngine.popen_uci(SF_PATH)
sf.configure({'UCI_LimitStrength': True, 'UCI_Elo': SF_ELO})
print(f'Stockfish open at ELO {SF_ELO}')

prev_weights   = None
decisive_count = 0
engine_wins    = 0
engine_losses  = 0

print(f'MCTS v4 SF Teacher  RUN_ID={RUN_ID}  simulations={N_SIMULATIONS}  '
      f'SF_ELO={SF_ELO}  games {start_game}-{start_game + N_GAMES - 1}')
print('-' * 90)

try:
    for game_idx in range(N_GAMES):
        game_number  = start_game + game_idx
        engine_color = chess.WHITE if (game_number % 2 == 1) else chess.BLACK
        t0           = time.time()

        examples, board, n_halfmoves, terminated_by, resigned_turn = play_game_vs_sf(
            net, sf, engine_color
        )
        replay_buffer.extend(examples)
        duration = time.time() - t0

        game_policy_loss = eval_policy_loss(net, examples)

        loss, v_loss, p_loss = None, None, None
        if len(replay_buffer) >= BATCH_SIZE:
            batch                = random.sample(replay_buffer, BATCH_SIZE)
            loss, v_loss, p_loss = train_step(net, optimizer, batch)
            scheduler.step()

        outcome_str = log_game(
            game_number, board, n_halfmoves, duration, game_policy_loss,
            engine_color, terminated_by=terminated_by, resigned_turn=resigned_turn
        )
        log_feature_stats(game_number, outcome_str, examples)

        engine_color_str = 'W' if engine_color == chess.WHITE else 'B'
        engine_won  = (outcome_str == '1-0' and engine_color == chess.WHITE) or \
                      (outcome_str == '0-1' and engine_color == chess.BLACK)
        engine_lost = (outcome_str == '0-1' and engine_color == chess.WHITE) or \
                      (outcome_str == '1-0' and engine_color == chess.BLACK)
        if outcome_str != '1/2-1/2':
            decisive_count += 1
        if engine_won:
            engine_wins += 1
        if engine_lost:
            engine_losses += 1

        if game_number % CHECKPOINT_EVERY == 0:
            prev_weights = log_weights(game_number, prev_weights)
            eng.save_model(net, MODEL_PATH)

        loss_str   = f'{loss:.4f}' if loss is not None else '  n/a '
        p_loss_str = f'{p_loss:.4f}' if p_loss is not None else '  n/a '
        dec_pct    = decisive_count / (game_idx + 1) * 100
        print(f'G{game_number:4d}({engine_color_str}) | {outcome_str:7s} | '
              f'{n_halfmoves:3d} ply | {duration:5.1f}s | '
              f'loss={loss_str} p={p_loss_str} | buf={len(replay_buffer):5d} | '
              f'E:{engine_wins}W/{engine_losses}L | decisive={dec_pct:4.1f}%')

finally:
    sf.quit()
    print('Stockfish closed.')

print('-' * 90)
print(f'Done. Model saved to {MODEL_PATH}')
print(f'Engine record vs SF{SF_ELO}: {engine_wins}W / {engine_losses}L / '
      f'{N_GAMES - engine_wins - engine_losses}D')
con.close()